# wrangle / portfolio / utility

**Intent**
- Estimates `portfolio_utility_value` from `portfolio_utility_wealth`, and must:
  - Give constant relative risk aversion utility of a wealth multiple.
  - Rank portfolios and decisions by what they are worth to you, not by their average.
  - Hold the rmse on `validation` and `backtest` to 0.001.

**Data**
- Personal data is git-ignored in `data/`, with synthetic placeholders committed.
- `training` is ten wealth multiples with their utility, `validation` is five larger, and `backtest` is ten across the range.
- `backtest` is held back for acceptance.
- `golden` is multiples from half to ten, then a request with the input missing.

**Status**
- The risk aversion is recovered from synthetic utilities, and the real input is your own choices.

**Future**
- Overall: the work still to do before the first release:
  - Add `query_postgres` to `collect_run` and load the real series in place of the synthetic ones.
  - Elicit risk aversion from certainty equivalents of 50/50 gambles, for you and your partner.
  - Ask twice, a year apart, since a risk aversion that moves with the day is noise.
  - Report a low, central and high risk aversion, not a point.
  - Fix the limits against real data, then register and review before any release.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from mlmodels import setup_workspace

ws = setup_workspace("wrangle/portfolio/utility")


GOLDEN = "portfolio_utility_golden"

## Collect ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-collect))

In [ ]:
from math import log

from mlmodels import collect_run


def utility(wealth, gamma=3.0):
    return log(wealth) if gamma == 1.0 else wealth ** (1.0 - gamma) / (1.0 - gamma)


def labelled(wealths):
    return {ws.inputs[0]: wealths, ws.outputs[0]: [round(utility(wealth), 6) for wealth in wealths]}


collect_run(
    ws,
    1000,
    label="Ten synthetic wealth multiples with their utility",
    name="portfolio_utility_training",
    source="builder",
    build=lambda: labelled([0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]),
)
collect_run(ws, 1000, label="Five larger synthetic wealth multiples", name="portfolio_utility_validation", source="builder", build=lambda: labelled([6.0, 7.0, 8.0, 9.0, 10.0]))
collect_run(
    ws, 1000, label="Ten synthetic wealth multiples across the range", name="portfolio_utility_backtest", source="builder", build=lambda: labelled([0.6, 0.9, 1.1, 1.7, 2.2, 3.3, 3.7, 4.5, 4.8, 12.0])
)

## Prepare ([6.7.3](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#673-prepare))

In [ ]:
from mlmodels import prepare_split

prepare_split(ws, training=ws.run(1000, "portfolio_utility_training"), validation=ws.run(1000, "portfolio_utility_validation"), backtest=ws.run(1000, "portfolio_utility_backtest"))
print("training", ws.training.height, "validation", ws.validation.height, "backtest", ws.backtest.height)

## Train ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-train))

In [ ]:
from scipy.optimize import curve_fit

fitted_gamma = curve_fit(lambda wealth, gamma: wealth ** (1.0 - gamma) / (1.0 - gamma), ws.training[ws.inputs[0]].to_numpy(), ws.training[ws.outputs[0]].to_numpy(), p0=[2.0], bounds=(1.5, 10.0))[0][0]
ws.params = {"gamma": round(float(fitted_gamma), 2)}
print("fitted", ws.params)
assert ws.params == {"gamma": 3.0}, "the fit did not recover the risk aversion"

## Define ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-define))

In [ ]:
from mlmodels import define_model

GAMMA = ws.params["gamma"]


def utility_model(wealth):
    from math import log

    return log(wealth) if GAMMA == 1.0 else wealth ** (1.0 - GAMMA) / (1.0 - GAMMA)


define_model(ws, model=1000, flavour="pyfunc", source=utility_model, label="Constant relative risk aversion utility, least squares risk aversion", constants={"GAMMA": ws.params["gamma"]})

## Evaluate ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-evaluate))

In [ ]:
from mlmodels import evaluate_gate, evaluate_metrics, evaluate_model, evaluate_models

LIMITS = dict.fromkeys(ws.outputs, 0.001)


def acceptance_checks():
    checks = []
    for split in ("validation", "backtest"):
        frame = getattr(ws, split)
        predicted = evaluate_model(ws, 1000, split)
        checks.extend(
            {
                "split": split,
                "gate": split == "backtest",
                "output": output,
                "name": "rmse",
                "value": evaluate_metrics(frame[output], predicted[output])["rmse"],
                "limit": LIMITS[output],
                "direction": "max",
            }
            for output in ws.outputs
        )
    return checks


evaluate_gate(ws, 1000, acceptance_checks)
evaluate_models(ws)

## Package ([6.7.8](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#678-package))

In [ ]:
from mlmodels import package_model

golden = {ws.inputs[0]: [0.5, 1.0, 1.5, 2.0, 5.0, 10.0, None], "expect": ["ok"] * 6 + ["error"]}
package_model(ws, 1000, golden, target={"kind": "local"})